# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedAmmarSharif/Flyrank-ML-Internship-Work/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

My lane is Refresh / Content Opportunity Scoring.

I will use Logistic Regression as my first machine-learning model because this is a binary classification problem. The target identifies whether a content page is declining or not declining.

Logistic Regression is a suitable starting point because it is relatively simple, interpretable, and produces probabilities that can be used to rank pages for human review.

I will compare its ranking performance against my Week-4 rule-based baseline using Precision@50. The goal is to determine whether machine learning provides better prioritization, not simply to use a more complex model.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design


I will use a client-grouped train/test split with 80% of clients for training and 20% for testing.

Pages from the same client will remain in only one split. This prevents the model from learning client-specific patterns from training data and being tested on pages from that same client.

I will use a fixed random seed to make the split reproducible. The test set will remain separate until model evaluation.

The target will be `is_declining_label`, derived from `trend_direction`. The target and `trend_pct` will not be used as input features.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

from google.colab import files
from sklearn.model_selection import GroupShuffleSplit

uploaded = files.upload()
df = pd.read_csv(next(iter(uploaded)))

print("Dataset shape:", df.shape)

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_id"])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("Training clients:", train_df["client_id"].nunique())
print("Testing clients:", test_df["client_id"].nunique())

print("Overlapping clients:", len(set(train_df["client_id"]) & set(test_df["client_id"])))

Saving content_refresh_anonymized (1).csv to content_refresh_anonymized (1) (1).csv
Dataset shape: (30000, 44)
Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Overlapping clients: 0


## 3. Train + compare vs my baseline

I will train Logistic Regression using the training clients only. The model will estimate the probability that each test page is declining.

I will compare the model against my Week-4 baseline rule on the same held-out test clients using Precision@50.

The baseline uses the same original rule: pages must have at least 90 days since their last update and at least 500 impressions. Their score is their impressions when both conditions are met, otherwise zero.

Precision@50 measures how many of the top 50 recommended pages are actually declining. This comparison will show whether the model improves prioritization over the simple rule.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]
y_train = train_df["is_declining_label"]
y_test = test_df["is_declining_label"]

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
model_scores = model.predict_proba(X_test)[:, 1]

baseline_scores = (
    (test_df["days_since_last_update"] >= 90).astype(int)
    * (test_df["impressions_90d"] >= 500).astype(int)
    * test_df["impressions_90d"]
)

def precision_at_50(scores, actual):
    top_indices = np.argsort(-np.asarray(scores), kind="stable")[:50]
    return np.asarray(actual)[top_indices].mean()

model_p50 = precision_at_50(model_scores, y_test)
baseline_p50 = precision_at_50(baseline_scores, y_test)

comparison = pd.DataFrame({
    "Method": ["Week-4 Baseline", "Logistic Regression"],
    "Precision@50": [baseline_p50, model_p50]
})

display(comparison)

print("Model trained successfully.")
print("Test pages:", len(test_df))
print("Evaluation metric: Precision@50")

,Method,Precision@50
0,Week-4 Baseline,0.3
1,Logistic Regression,0.6


Model trained successfully.
Test pages: 6163
Evaluation metric: Precision@50


### Model comparison results

Logistic Regression achieved Precision@50 of 0.60, compared with 0.30 for the Week-4 baseline.

This means 30 of the model's top 50 recommended pages were declining, compared with 15 for the baseline.

The model improved Precision@50 by 0.30 on the same held-out test clients. This is a promising improvement in prioritization, although further validation is needed before treating it as reliable across all clients.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.metrics import confusion_matrix, classification_report

test_results = test_df.copy()

test_results["predicted_label"] = model.predict(X_test)
test_results["predicted_probability"] = model_scores

false_positives = test_results[
    (test_results["predicted_label"] == 1) &
    (test_results["is_declining_label"] == 0)
]

false_negatives = test_results[
    (test_results["predicted_label"] == 0) &
    (test_results["is_declining_label"] == 1)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, model.predict(X_test)))

print("\nClassification report:")
print(classification_report(y_test, model.predict(X_test)))

print("\nFalse positive examples:")
display(false_positives[["content_id", "impressions_90d", "ctr", "predicted_probability"]].head(5))

print("\nFalse negative examples:")
display(false_negatives[["content_id", "impressions_90d", "ctr", "predicted_probability"]].head(5))

coefficients = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": model.named_steps["classifier"].coef_[0]
})

coefficients["absolute_importance"] = (
    coefficients["coefficient"].abs()
)

display(coefficients.sort_values("absolute_importance", ascending=False))

False positives: 1513
False negatives: 1314

Confusion matrix:
[[1501 1513]
 [1314 1835]]

Classification report:
              precision    recall  f1-score   support

           0       0.53      0.50      0.52      3014
           1       0.55      0.58      0.56      3149

    accuracy                           0.54      6163
   macro avg       0.54      0.54      0.54      6163
weighted avg       0.54      0.54      0.54      6163


False positive examples:


,content_id,impressions_90d,ctr,predicted_probability
13,content_a5a2fbc76336,307,0.00,0.613017
36,content_bce275871a25,371,1.35,0.549033
56,content_dcebfd222b10,16,0.00,0.606554
64,content_685de0e3b7cb,2639,0.11,0.613066
78,content_dea0d86223f3,59,0.00,0.655636



False negative examples:


,content_id,impressions_90d,ctr,predicted_probability
1,content_a1fb4e703a9e,15320,0.05,0.378497
23,content_2da6ae9d0882,297,0.34,0.334826
54,content_ff8ea1364b59,170,0.00,0.342266
58,content_caff51984338,71,0.00,0.345763
81,content_16788821b64a,320,0.31,0.338635


,feature,coefficient,absolute_importance
5,content_age_days,-0.381368,0.381368
6,days_since_last_update,0.226637,0.226637
3,ctr,-0.181633,0.181633
1,clicks_90d,-0.169102,0.169102
7,word_count,0.063566,0.063566
0,impressions_90d,0.022442,0.022442
2,avg_position,-0.018005,0.018005
4,engagement_rate,-0.008489,0.008489


## 4. Errors and interpretation

### Classification errors

The model produced 1,513 false positives and 1,314 false negatives on the held-out test set.

False positives are pages predicted as declining when their actual label is not declining. False negatives are declining pages that the model failed to identify.

At the default classification threshold of 0.5, the model achieved 54% accuracy, 55% precision, and 58% recall for the declining class. This shows that the model still makes substantial classification errors.

### Feature interpretation

The largest absolute Logistic Regression coefficient belonged to content_age_days (-0.381), followed by days_since_last_update (+0.227), ctr (-0.182), and clicks_90d (-0.169).

These coefficients describe relationships learned from the training data. They do not prove that changing a feature will cause a page to stop declining.

### Overall interpretation

Logistic Regression achieved Precision@50 of 0.60 compared with 0.30 for the Week-4 baseline on the same held-out test clients.

Although the ranking result is promising, the classification report shows that the model still makes many errors. It should therefore be used as a decision-support tool for human review rather than an automatic content-refresh decision.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.